# 전철역 리스트(UrbanRailwayLinespecificStation) → DuckDB 적재

공공데이터포털 `UrbanRailwayLinespecificStation/getUrbanRailwayLinespecificStation` API.

- `opr_ymd` (운행일자): **오늘 - 1개월**
- 시도 필터 없음 → 전국 한 번에 반환
- 테이블: `railway_station` (PK = sarea_nm + rte_id + sttn_id 복합키)
- ⚠️ 환승역(한 역이 여러 노선에 속함)이라 sttn_id 단독으로는 PK 불가
- ⚠️ 이 API의 rte_id는 8자리('00000002'), railway_line의 rte_id는 3자리('003')로 포맷이 다름 — JOIN 시 zero-pad 정규화 필요

In [14]:
# ============================================================
# 셀 1 - 환경 설정 + DB 헬퍼 정의
# ============================================================
import os
import time
import requests
import duckdb
import pandas as pd
from pathlib import Path
from contextlib import contextmanager
from datetime import date
from dateutil.relativedelta import relativedelta
from urllib.parse import unquote
from dotenv import load_dotenv

load_dotenv()
SERVICE_KEY = unquote(os.environ["MY_SERVICE_KEY"])

DB_PATH = "data/seoul.duckdb"
Path("data").mkdir(exist_ok=True)


@contextmanager
def db_open(read_only: bool = False):
    """짧게 연결하고 무조건 닫는 컨텍스트 매니저 (예외 시에도 안전)."""
    con = duckdb.connect(DB_PATH, read_only=read_only)
    try:
        yield con
    finally:
        con.close()


print(f"SERVICE_KEY 로드됨 (길이 {len(SERVICE_KEY)})")
print(f"DB: {DB_PATH} (헬퍼 준비됨)")

SERVICE_KEY 로드됨 (길이 88)
DB: data/seoul.duckdb (헬퍼 준비됨)


In [15]:
# ============================================================
# 셀 2 - Pydantic 모델 (공공데이터 표준 응답 + RailwayStationItem)
# ============================================================
from typing import Generic, TypeVar, Optional
from pydantic import BaseModel, field_validator

T = TypeVar("T", bound=BaseModel)


class Header(BaseModel):
    resultCode: str
    resultMsg: str


class Items(BaseModel, Generic[T]):
    item: list[T] = []

    @field_validator("item", mode="before")
    @classmethod
    def _coerce_to_list(cls, v):
        if v is None or v == "":
            return []
        if isinstance(v, dict):
            return [v]
        return v


class Body(BaseModel, Generic[T]):
    items: Items[T] = Items[T]()
    dataType: str | None = None
    pageNo: int
    numOfRows: int
    totalCount: int


class Response_(BaseModel, Generic[T]):
    header: Header
    body: Body[T]


class PublicDataResponse(BaseModel, Generic[T]):
    Response: Response_[T]


class RailwayStationItem(BaseModel):
    """전철역 한 건.

    ⚠️ API가 일부 행의 sarea_nm / rte_id / sttn_id까지 null로 반환 → 전부 Optional로 방어
       Pydantic에서는 일단 통과시키고, 적재 셀에서 PK가 null인 행을 걸러냄.

    PK (적재 시): (sarea_nm, rte_id, sttn_id) — null 값 제거 후 삽입
      - 같은 sttn_id가 여러 노선(환승역)에 속할 수 있음
      - sttn_seq: 노선 내 역 순서
      - sttn_nm 포맷: '민락역 [부산지하철2호선]' 처럼 노선명이 꺾쇠로 붙어 있음
    """
    # opr_ymd만 필수 (파라미터로 넘긴 값이 그대로 돌아오는 필드)
    opr_ymd: str

    # 원래 PK 후보지만 null이 올 수 있어 전부 Optional
    sarea_nm: Optional[str] = None
    rte_id: Optional[str] = None
    sttn_id: Optional[str] = None

    # 나머지 Optional
    rte_nm: Optional[str] = None
    sttn_nm: Optional[str] = None
    sttn_seq: Optional[int] = None


print("모델 로드 완료")

모델 로드 완료


In [16]:
# ============================================================
# 셀 3 - fetch 함수 (NO_DATA_FOUND 에러 처리 포함)
# ============================================================
URBAN_STATION_URL = "https://apis.data.go.kr/1613000/UrbanRailwayLinespecificStation/getUrbanRailwayLinespecificStation"


def fetch_all_pages(
    url: str,
    item_model: type[T],
    extra_params: dict | None = None,
    num_rows: int = 1000,
) -> list[T]:
    """공공데이터 표준 응답 API 전체 페이지 수집. NO_DATA_FOUND는 빈 결과로 취급."""
    params_base = {
        "serviceKey": SERVICE_KEY,
        "numOfRows": num_rows,
        "dataType": "JSON",
        **(extra_params or {}),
    }
    all_items: list[T] = []
    page = 1
    while True:
        res = requests.get(url, params={**params_base, "pageNo": page}, timeout=30)
        res.raise_for_status()
        payload = res.json()

        # 에러 응답 선처리
        if "Error" in payload:
            err = payload["Error"]
            code = err.get("code")
            msg = err.get("message", "")
            if msg == "NO_DATA_FOUND" or code == "50":
                return all_items
            raise RuntimeError(f"API error: {code} {msg}")

        # Pydantic 검증
        parsed = PublicDataResponse[item_model].model_validate(payload)
        header = parsed.Response.header
        if header.resultCode not in ("00", "200"):
            raise RuntimeError(f"API error: {header.resultCode} {header.resultMsg}")

        body = parsed.Response.body
        items = body.items.item
        all_items.extend(items)
        print(f"  page {page}: +{len(items)}건 (누적 {len(all_items)}/{body.totalCount})")

        if len(all_items) >= body.totalCount or not items:
            break
        page += 1
        time.sleep(0.15)

    return all_items


print("fetch 함수 준비 완료")

fetch 함수 준비 완료


In [17]:
# ============================================================
# 셀 4 - 전국 전철역 수집
# ============================================================
# opr_ymd = 오늘 - 1개월. 시도 루프 불필요 (API가 한 번에 전국 반환).
# 전국 전철역은 수천 건 규모 예상 → 여러 페이지 순회.

opr_ymd = (date.today() - relativedelta(months=1)).strftime("%Y%m%d")
print(f"운행일자 opr_ymd = {opr_ymd}")

stations = fetch_all_pages(
    URBAN_STATION_URL,
    RailwayStationItem,
    extra_params={"opr_ymd": opr_ymd},
)

station_df = pd.DataFrame([s.model_dump() for s in stations])
print(f"\n총 {len(station_df)}건 수집")
print(station_df.head(10))

운행일자 opr_ymd = 20260322
  page 1: +1000건 (누적 1000/1582)
  page 2: +582건 (누적 1582/1582)

총 1582건 수집
    opr_ymd sarea_nm rte_id sttn_id rte_nm             sttn_nm  sttn_seq
0  20260322      수도권    001    0150    1호선            서울 [1호선]         1
1  20260322      수도권    001    0151    1호선            시청 [1호선]         2
2  20260322      수도권    001    0152    1호선            종각 [1호선]         3
3  20260322      수도권    001    0153    1호선          종로3가 [1호선]         4
4  20260322      수도권    001    0154    1호선          종로5가 [1호선]         5
5  20260322      수도권    001    0155    1호선           동대문 [1호선]         6
6  20260322      수도권    001    0156    1호선           신설동 [1호선]         8
7  20260322      수도권    001    0157    1호선           제기동 [1호선]         9
8  20260322      수도권    001    0158    1호선  청량리(서울시립대입구) [1호선]        10
9  20260322      수도권    001    0159    1호선           동묘앞 [1호선]         7


In [18]:
# ============================================================
# 셀 5 - DuckDB 적재 (railway_station 테이블, 복합 PK)
# ============================================================
# 1) sarea_nm 보정: API 품질 이슈로 null인 경우가 있음 (주로 수도권 경춘선 등)
#    → railway_line에서 rte_nm 기준으로 lookup, 매칭 안 되는 건 '수도권' fallback
#    ⚠️ rte_id는 API마다 포맷 다름(line=3자리, station=8자리) → rte_nm으로 매칭

with db_open(read_only=True) as con:
    rte_nm_to_sarea = con.execute("""
        SELECT DISTINCT rte_nm, sarea_nm
        FROM railway_line
        WHERE rte_nm IS NOT NULL
    """).df()

null_sarea_cnt = station_df["sarea_nm"].isna().sum()
if null_sarea_cnt > 0:
    print(f"ℹ️  sarea_nm null {null_sarea_cnt}건 → railway_line.rte_nm 기준으로 보정 시도")

    # rte_nm 매칭으로 sarea_nm 채우기
    lookup = rte_nm_to_sarea.rename(columns={"sarea_nm": "_sarea_fill"})
    station_df = station_df.merge(lookup, on="rte_nm", how="left")
    station_df["sarea_nm"] = station_df["sarea_nm"].fillna(station_df["_sarea_fill"])
    station_df = station_df.drop(columns=["_sarea_fill"])

    # 그래도 남은 null → '수도권' fallback (관측된 null 케이스는 모두 수도권이었음)
    still_null = station_df["sarea_nm"].isna().sum()
    if still_null > 0:
        print(f"    lookup 실패 {still_null}건 → '수도권' fallback")
        station_df["sarea_nm"] = station_df["sarea_nm"].fillna("수도권")

# 2) 그래도 rte_id / sttn_id가 null인 행은 PK 구성 불가 → 제거
total_before = len(station_df)
station_df_clean = station_df.dropna(subset=["sarea_nm", "rte_id", "sttn_id"])
null_dropped = total_before - len(station_df_clean)
if null_dropped > 0:
    print(f"⚠️  rte_id 또는 sttn_id null {null_dropped}건 제외")

# 3) 중복 (복합키) 제거
before = len(station_df_clean)
station_df_dedup = station_df_clean.drop_duplicates(subset=["sarea_nm", "rte_id", "sttn_id"])
after = len(station_df_dedup)
if before != after:
    print(f"⚠️  중복 (sarea_nm, rte_id, sttn_id) {before - after}건 제거")

with db_open() as con:
    con.execute("""
    CREATE TABLE IF NOT EXISTS railway_station (
        sarea_nm   TEXT,
        rte_id     TEXT,
        sttn_id    TEXT,
        opr_ymd    TEXT,
        rte_nm     TEXT,
        sttn_nm    TEXT,
        sttn_seq   INTEGER,
        PRIMARY KEY (sarea_nm, rte_id, sttn_id)
    )
    """)

    con.execute("DELETE FROM railway_station")
    con.register("station_view", station_df_dedup)
    con.execute("""
    INSERT INTO railway_station
    SELECT sarea_nm, rte_id, sttn_id, opr_ymd, rte_nm, sttn_nm, sttn_seq
    FROM station_view
    """)
    con.unregister("station_view")

    cnt = con.execute("SELECT COUNT(*) FROM railway_station").fetchone()[0]

print(f"적재 완료: {cnt}건")

ℹ️  sarea_nm null 6건 → railway_line.rte_nm 기준으로 보정 시도
⚠️  중복 (sarea_nm, rte_id, sttn_id) 479건 제거
적재 완료: 1103건


In [19]:
# ============================================================
# 셀 6 - 검증
# ============================================================
with db_open(read_only=True) as con:
    print("=== 서비스 지역별 역 수 ===")
    print(con.execute("""
        SELECT sarea_nm, COUNT(*) AS stn_cnt
        FROM railway_station
        GROUP BY sarea_nm ORDER BY stn_cnt DESC
    """).df())

    print("\n=== 수도권 노선별 역 수 TOP 10 ===")
    print(con.execute("""
        SELECT rte_nm, COUNT(*) AS stn_cnt
        FROM railway_station
        WHERE sarea_nm = '수도권'
        GROUP BY rte_nm ORDER BY stn_cnt DESC
        LIMIT 10
    """).df())

    print("\n=== 환승역 상위 (수도권, 3개 노선 이상) ===")
    print(con.execute("""
        WITH pure AS (
            SELECT regexp_extract(sttn_nm, '^[^\\s\\[]+', 0) AS stn_base, rte_nm
            FROM railway_station
            WHERE sarea_nm = '수도권'
        )
        SELECT stn_base, COUNT(DISTINCT rte_nm) AS line_cnt
        FROM pure
        GROUP BY stn_base
        HAVING line_cnt >= 3
        ORDER BY line_cnt DESC
        LIMIT 10
    """).df())

=== 서비스 지역별 역 수 ===
  sarea_nm  stn_cnt
0      수도권      784
1   부산·울산권      147
2      대구권      119
3      대전권       33
4      광주권       20

=== 수도권 노선별 역 수 TOP 10 ===
  rte_nm  stn_cnt
0    1호선       92
1    7호선       62
2  경의중앙선       57
3    5호선       56
4  수인분당선       54
5    4호선       51
6    2호선       51
7    3호선       44
8    6호선       39
9    9호선       38

=== 환승역 상위 (수도권, 3개 노선 이상) ===
       stn_base  line_cnt
0          김포공항         5
1            서울         5
2            공덕         4
3     왕십리(성동구청)         4
4          홍대입구         3
5            회기         3
6  청량리(서울시립대입구)         3
7            수서         3
8            대곡         3
9           연신내         3
